In [ ]:
from preprocessing import load_data, prep_and_split_data
from lgb import train_and_evaluate_lgb_classifier, train_and_evaluate_lgb_regressor
from constants import TRAIN_FILENAME, TARGET_CLASS, CURRENT_PRICE
from ft_eng import engineer_features
import pandas as pd

train_df = load_data(TRAIN_FILENAME)
train_df = engineer_features(train_df)

X_train, X_val, y_train_class, y_val_class, y_train_price, y_val_price = (
    prep_and_split_data(train_df)
)

# grab the validation IDs for later
val_ids = train_df.loc[X_val.index, "ID"]


classifier_model, _ = train_and_evaluate_lgb_classifier(
    X_train, y_train_class, X_val, y_val_class
)

regressor_model, _ = train_and_evaluate_lgb_regressor(
    X_train, y_train_price, X_val, y_val_price
)


# predict probabilities of drop from p40 to p50 and expected returns (at p50)
val_prob_drop = classifier_model.predict_proba(X_val)[:, 0]
val_pred_returns = regressor_model.predict(X_val)

ev_from_sell = val_prob_drop * (X_val[CURRENT_PRICE] - val_pred_returns)

eval_df = pd.DataFrame(
    {
        "ID": val_ids,
        "Current_Price": X_val[CURRENT_PRICE],
        "Final_Price": y_val_price,
        "EV_From_Sell": ev_from_sell,
    }
)

# assign 0 by default, find top 800 by order of EV, update them to 1
eval_df["is_sold"] = 0
top_800_idx = eval_df.nlargest(800, "EV_From_Sell").index
eval_df.loc[top_800_idx, "is_sold"] = 1

# calculate portfolio returns
# p_r = sum of selling all chosen stocks at p40, and selling all unchosen stocks at p50
portfolio_return = (
    eval_df.loc[eval_df["is_sold"] == 1, "Current_Price"].sum()
    + eval_df.loc[eval_df["is_sold"] == 0, "Final_Price"].sum()
)
print(f"portfolio return: {portfolio_return}")


test_df = load_data("test.csv")
test_df = engineer_features(test_df)

test_ids = test_df["ID"]
test_df = test_df.drop(columns=["ID"])

# same thing as above but now with test data
test_prob_drop = classifier_model.predict_proba(test_df)[:, 0]
test_pred_returns = regressor_model.predict(test_df)
test_ev_from_sell = test_prob_drop * (test_df[CURRENT_PRICE] - test_pred_returns)

submission = pd.DataFrame(
    {"ID": test_ids, "EV_From_Sell": test_ev_from_sell, TARGET_CLASS: 0}
)

top_4000_idx = submission.nlargest(4000, "EV_From_Sell").index
submission.loc[top_4000_idx, TARGET_CLASS] = 1

# drop ev col for submission
submission = submission[["ID", TARGET_CLASS]]
submission.to_csv("submission_new.csv", index=False)

# just in case
num_sold = submission[TARGET_CLASS].sum()
print(f"Total stocks sold: {num_sold}")

Training with params: {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
Validation Accuracy: 0.5355
Training Regressor with params: {'n_estimators': 300, 'learning_rate': 0.025, 'max_depth': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
Validation MSE: 225.097762
Local Portfolio Return: 200370.42
Total stocks sold in submission: 4000


In [2]:
from evaluation import show_top_features, show_bottom_features

feature_names = [col for col in X_train.columns]

ft_df = show_top_features(
    regressor_model,
    feature_names,
)

b_ft_df = show_bottom_features(regressor_model, feature_names)


TOP 10 STRONGEST FEATURES
         Feature  Importance
             p40        1104
    Last_5_ratio         385
    trend_last_5         373
overall_change_r         343
           Std_5         296
          Std_10         270
              p1         258
              p2         250
             p38         212
              p3         206

TOP 10 WEAKEST FEATURES
Feature  Importance
    p36          66
    p26          71
    p27          71
    p29          73
    p34          73
    p22          98
    p30         101
    p35         102
    p31         108
    p24         110
